## Introduction

Music has always played a central role in my life. I am deeply passionate about discovering and experiencing music, regularly attending live concerts and performances, and maintaining a personal music collection composed of both vinyl records and CDs. Beyond listening, I have been interested in understanding how my musical preferences have evolved over time: what I listen to most, when I listen, and how my habits change depending on context.

Musical taste is often strongly shaped during adolescence and early adulthood, when exposure to new artists and genres tends to be at its peak. As people grow older, discovery can gradually slow, and many listeners fall into what is sometimes described as music paralysis: a tendency to stop exploring new artists and instead return to familiar music. Rather than accepting this as an inevitable outcome, I am interested in understanding whether my own listening behavior has remained open to discovery and aligned with evolving music trends over the past decade. 

By analyzing long-term Spotify listening data, this project seeks to evaluate whether my habits reflect ongoing exploration or a gradual shift toward repetition, using data to measure how musical curiosity changes over time. It also explores how my musical taste has evolved and expanded over time, reflecting personal experiences and life transitions. In particular, moving to the United States at the age of eighteen introduced new cultural environments, social contexts, and musical influences that may have reshaped my listening habits. At the same time, music provides a way to remain connected to my cultural origins, raising questions about how identity is preserved or transformed through listening choices.
This analysis therefore treats streaming history not simply as usage data, but as a longitudinal record of musical identity: examining how exploration, cultural connection, and changing environments interact to shape what I listen to and how my relationship with music continues to develop over time.

This project transforms ten years of Spotify streaming history into an analytical database and interactive Tableau dashboard. By structuring raw streaming data into an analytics-ready dataset, the project combines personal curiosity with data analytics practices to explore listening behavior through measurable patterns.

The workflow follows an end-to-end analytics approach, beginning with raw JSON data ingestion and transformation within MongoDB, where the streaming history is cleaned, modeled, and enriched for analysis. The resulting dataset is optimized for performance and scalability before being visualized in Tableau, which serves as the exploration and storytelling layer of the project.

Together, these analyses aim to reveal how personal taste develops over time while demonstrating an end-to-end data analytics workflow, from raw data collection to interactive visualization.

## MongoDB Database Creation and Data Pipeline

### Data Source

The first step consists of downloading Spotify's Extended Streaming History, which contains the complete listening activity associated with a user account. The dataset is provided by Spotify upon request and delivered as multiple JSON files, each containing historical streaming events.

Each record represents a single listening event and includes metadata such as:

* ts: Date and time of when the stream ended in UTC format.
* username: Spotify username.
* platform: Platform used when streaming the track (e.g. Android OS, Google Chromecast).
* ms_played: Milliseconds the track was played.
* conn_country: Country code of the country where the stream was played.
* ip_addr: IP address used when streaming the track.
* master_metadata_track_name: Name of the track.
* master_metadata_album_artist_name: Name of the artist, band or podcast.
* master_metadata_album_album_name: Name of the album of the track.
* spotify_track_uri: A Spotify Track URI, that is identifying the unique music track.
* episode_name: Name of the episode of the podcast.
* episode_show_name: Name of the show of the podcast.
* spotify_episode_uri: A Spotify Episode URI, that is identifying the unique podcast episode.
* audiobook_title:
* audiobook_uri:
* audiobook_chapter_uri:
* audiobook_chapter_title:
* reason_start: Reason why the track started (e.g. previous track finished or you picked it from the playlist).
* reason_end: Reason why the track ended (e.g. the track finished playing or you hit the next button).
* shuffle: Whether shuffle mode was used when playing the track.
* skipped: Whether the user skipped to the next song.
* offline: Whether the track was played in offline mode.
* offline_timestamp: Timestamp of when offline mode was used, if it was used.
* incognito_mode: Whether the track was played during a private session.

Since the analytical scope of this project focuses exclusively on music listening behavior, all podcast and audiobook entries are removed during preprocessing.

In [1]:
# Import the MongoClient class from the pymongo library
from pymongo import MongoClient
# Create a MongoClient object and specify the connection URL. Currently running locally
client = MongoClient('mongodb://localhost:27017/')
# Create the database
db_name = "spotifyDatabase"
# List all databases
databases = client.list_database_names()
if db_name in databases:
    # Drop the database
    client.drop_database(db_name)
db = client[db_name]

### Data Pipeline
### 1. Raw Data Collection

All JSON files are first loaded without modification into a MongoDB collection named `streamingRawData`.
This layer serves as a permanent source of truth and allows transformations to be reproduced or updated without re-importing data.
Process performed:
- Load all JSON streaming history files
- Append records into a single collection
- Preserve original schema and timestamps

In [2]:
import json
import os
# Create the raw streaming history collection
raw_collection_name = "streamingRawData"
raw_collection = db[raw_collection_name]
# Directory containing JSON files with streaming history
json_directory = os.path.expanduser("~/Desktop/Dani/SpotifyDatabase/StreamingHistoryData")
# Function to find all json files in directory and subdirectories
def find_json_files(directory):
    json_files = []
    for root, _, files in os.walk(directory):
        for file in files:
            if file.endswith(".json"):
                json_files.append(os.path.join(root, file))
    return json_files
streaming_json_files = find_json_files(json_directory)
print(f"Found {len(streaming_json_files)} JSON files.")
# Iterate through all JSON files in the directory
for file_path in streaming_json_files:
    if file_path.endswith(".json"):  # Ensure it's a JSON file, this might not be necessary
        with open(file_path, 'r', encoding='utf-8') as file:
            # Load JSON data
            data = json.load(file)
            # Insert data into MongoDB
            if isinstance(data, list):  # If the JSON file contains an array of documents
                raw_collection.insert_many(data)
            else:  # If the JSON file contains a single document
                raw_collection.insert_one(data)

Found 17 JSON files.


### 2. Analytics Collections

A second collection, `streamingAnalytics`, is generated using MongoDB aggregation pipelines. This collection contains a cleaned and feature-engineered dataset specifically designed for Tableau consumption.

A cleaning pipeline is applied to remove irrelevant or non-analytical information.
The following transformations are performed:
- Remove podcast and audiobook entries to maintain a music-only dataset
- Remove unnecessary parameters such as IP address and username for privacy and analytical relevance
- Standardize field names for readability and consistency
- Convert timestamps into a uniform datetime format
- Filter invalid or extremely short playback events

In [3]:
# Create the streaming analytics collection
analytics_collection_name = "streamingAnalytics"
analytics_collection = db[analytics_collection_name]
# Copy the 'streamingRawData' to 'streamingAnalytics'
analytics_collection.insert_many(raw_collection.find())
# Make sure all entries were copied correctly
raw_count = db.streamingRawData.count_documents({})
analytics_count = db.streamingAnalytics.count_documents({})

print("Raw:", raw_count)
print("Analytics:", analytics_count)

Raw: 263144
Analytics: 263144


In [4]:
# Remove all entries that aren't songs
only_songs_query = {
    "$or": [
        {"master_metadata_track_name": None},
        {"master_metadata_album_artist_name": None}
    ]
}
analytics_collection.delete_many(only_songs_query)
# Remove unnecessary parameters
param_to_remove = {"episode_name": "", 
                   "episode_show_name": "", 
                   "spotify_episode_uri": "", 
                   "audiobook_title": "", 
                   "audiobook_uri": "", 
                   "audiobook_chapter_uri": "", 
                   "audiobook_chapter_title": "",
                    "username": "",
                    "ip_addr": "",
                    "spotify_episode_uri": "",
                    "offline_timestamp": "",
                    "offline": "",
                    "incognito_mode": "",
                    "platform": "",}
analytics_collection.update_many({}, {"$unset": param_to_remove})
# Remove plays under 5 seconds
analytics_collection.delete_many({
    "ms_played": {"$lt": 5000}
})
# Convert ms_played to sec_played for easier analysis and readability
analytics_collection.update_many(
    {},
    [
        {
            "$set": {
                "sec_played": {
                    "$divide": ["$ms_played", 1000]
                }
            }
        }
    ]
)
# Convert ts to time-based feaetures for easier analysis and readability
analytics_collection.update_many(
    {},
    [
        {
            "$set": {
                "played_at": {
                    "$toDate": "$ts"
                }
            }
        }
    ]
)
analytics_collection.update_many(
    {},
    [
        {
            "$set": {
                "year": {"$year": "$played_at"},
                "month": {"$month": "$played_at"},
                "day": {"$dayOfMonth": "$played_at"},
                "day_of_week": {"$dayOfWeek": "$played_at"},
                "hour_of_day": {"$hour": "$played_at"}
            }
        }
    ]
)
# Remove unnecessary time related parameters
param_to_remove = {"ts": "", 
                   "ms_played": "",}
analytics_collection.update_many({}, {"$unset": param_to_remove})
# Rename variables for easier access and readability
analytics_collection.update_many(
    {},
    {"$rename": {"master_metadata_track_name": "track_name"}}
)
analytics_collection.update_many(
    {},
    {"$rename": {"master_metadata_album_artist_name": "artist_name"}}
)
analytics_collection.update_many(
    {},
    {"$rename": {"master_metadata_album_album_name": "album_name"}}
)

UpdateResult({'n': 177207, 'nModified': 177207, 'ok': 1.0, 'updatedExisting': True}, acknowledged=True)

The `artistYearMetrics` collection contains aggregated listening statistics at the artist–year level and is designed to support longitudinal analysis of musical taste evolution. Each document represents the interaction between a single artist and a specific calendar year, aggregating individual streaming events to summarize how frequently and intensely that artist was listened to during that period.

Key metrics include total stream count, total listening time, number of unique tracks played, average playback duration, shuffled count, shuffle rate, skipped count, and skip rate. Additionally, a discovery indicator identifies whether a given year corresponds to the first appearance of an artist in the listening history, enabling analysis of artist discovery, listening consistency, and long-term engagement patterns.

In [5]:
# Compute artist-year level metrics and store in a new collection called 'artistYearMetrics'
db.streamingAnalytics.aggregate([
    {
        "$group": {
            "_id": {
                "year": "$year",
                "artist": "$artist_name"
            },
            "total_streams": {"$sum": 1},
            "total_sec_played": {"$sum": "$sec_played"},
            "unique_tracks": {"$addToSet": "$spotify_track_uri"},
            "skipped_count": {
                "$sum": {"$cond": ["$skipped", 1, 0]}
            },
            "shuffled_count": {
                "$sum": {"$cond": ["$shuffle", 1, 0]}
            },
            "avg_sec_played": {"$avg": "$sec_played"},
            "first_play_in_year": {"$min": "$played_at"}
        }
    },
    {
        "$project": {
            "_id": 0,
            "year": "$_id.year",
            "artist_name": "$_id.artist",
            "total_streams": 1,
            "total_sec_played": 1,
            "listening_hours": {
                "$divide": ["$total_sec_played", 3600]
            },
            "unique_tracks_played": {"$size": "$unique_tracks"},
            "skipped_count": 1,
            "skip_rate": {
                "$divide": ["$skipped_count", "$total_streams"]
            },
            "shuffled_count": 1,
            "shuffle_rate": {
                "$divide": ["$shuffled_count", "$total_streams"]
            },
            "avg_sec_played": 1,
            "first_play_in_year": 1
        }
    },
    {
        "$merge": {
            "into": "artistYearMetrics",
            "whenMatched": "replace",
            "whenNotMatched": "insert"
        }
    }
])
# Compute first year of streaming for each artists
db.artistYearMetrics.aggregate([
    {
        "$group": {
            "_id": "$artist_name",
            "first_year": {"$min": "$year"}
        }
    },
    {
        "$merge": {
            "into": "artistFirstYear",
            "whenMatched": "replace"
        }
    }
])
# Set flag for whether the artist was new in that year or not
db.artistYearMetrics.aggregate([
    {
        "$lookup": {
            "from": "artistFirstYear",
            "localField": "artist_name",
            "foreignField": "_id",
            "as": "artist_info"
        }
    },
    {"$unwind": "$artist_info"},
    {
        "$set": {
            "is_new_artist": {
                "$eq": ["$year", "$artist_info.first_year"]
            }
        }
    },
    {"$unset": "artist_info"},
    {"$merge": "artistYearMetrics"}
])
# Drop the artistFirstYear collection as it is no longer needed after merging the first year info into artistYearMetrics
db.artistFirstYear.drop()

The `songYearMetrics` collection contains aggregated listening statistics at the song–year level and is designed to analyze listening behavior and engagement with individual tracks over time. Each document represents the interaction between a single song and a specific calendar year, summarizing how frequently the track was played and how listening patterns evolved across years.

Key metrics include total stream count, total listening time, listening hours, average playback duration, shuffled count, shuffle rate, skipped count, skip rate, as well as the first and last playback timestamps within the year. A discovery indicator identifies whether a given year corresponds to the first appearance of a track in the listening history, enabling analysis of music discovery, repetition, and song longevity.

In [6]:
#Compute song-year level metrics and store in a new collection called 'songYearMetrics'
db.streamingAnalytics.aggregate([
    {
        "$group": {
            "_id": {
                "year": "$year",
                "track": "$track_name",
                "artist": "$artist_name",
                "album": "$album_name",
                "track_uri": "$spotify_track_uri"
            },

            "total_streams": {"$sum": 1},
            "total_sec_played": {"$sum": "$sec_played"},

            "skipped_count": {
                "$sum": {"$cond": ["$skipped", 1, 0]}
            },

            "shuffled_count": {
                "$sum": {"$cond": ["$shuffle", 1, 0]}
            },

            "avg_sec_played": {"$avg": "$sec_played"},

            "first_play_in_year": {"$min": "$played_at"},
            "last_play_in_year": {"$max": "$played_at"}
        }
    },
    {
        "$project": {
            "_id": 0,
            "year": "$_id.year",
            "track_name": "$_id.track",
            "artist_name": "$_id.artist",
            "album_name": "$_id.album",
            "spotify_track_uri": "$_id.track_uri",

            "total_streams": 1,
            "total_sec_played": 1,

            "listening_hours": {
                "$divide": ["$total_sec_played", 3600]
            },

            "skipped_count": 1,
            "skip_rate": {
                "$divide": ["$skipped_count", "$total_streams"]
            },

            "shuffled_count": 1,
            "shuffle_rate": {
                "$divide": ["$shuffled_count", "$total_streams"]
            },

            "avg_sec_played": 1,
            "first_play_in_year": 1,
            "last_play_in_year": 1
        }
    },
    {
        "$merge": {
            "into": "songYearMetrics",
            "whenMatched": "replace",
            "whenNotMatched": "insert"
        }
    }
])
# Compute first year of streaming for each song
db.songYearMetrics.aggregate([
    {
        "$group": {
            "_id": "$spotify_track_uri",
            "first_year": {"$min": "$year"}
        }
    },
    {
        "$merge": {
            "into": "songFirstYear",
            "whenMatched": "replace"
        }
    }
])
# Set flag for whether the song is new in that year or not
db.songYearMetrics.aggregate([
    {
        "$lookup": {
            "from": "songFirstYear",
            "localField": "spotify_track_uri",
            "foreignField": "_id",
            "as": "song_info"
        }
    },
    {"$unwind": "$song_info"},
    {
        "$set": {
            "is_new_song": {
                "$eq": ["$year", "$song_info.first_year"]
            }
        }
    },
    {"$unset": "song_info"},
    {"$merge": "songYearMetrics"}
])
# Drop the songFirstYear collection as it's no longer needed
db.songFirstYear.drop()

The `yearSummaryMetrics` collection aggregates listening activity at the calendar year level, summarizing total streams, listening hours, number of unique artists and tracks, skip and shuffle behavior, and discovery metrics. Each document represents one year and provides high-level insights into listening habits, exploration patterns, and music identity evolution over time, making it ideal for longitudinal visualization and analysis in Tableau.

In [7]:
# Compute year-level summary metrics and store in a new collection called 'yearSummaryMetrics'
db.streamingAnalytics.aggregate([
    {
        "$group": {
            "_id": "$year",
            "total_streams": {"$sum": 1},
            "total_sec_played": {"$sum": "$sec_played"},
            
            "skipped_count": {"$sum": {"$cond": ["$skipped", 1, 0]}},
            "shuffled_count": {"$sum": {"$cond": ["$shuffle", 1, 0]}},

            "unique_artists": {"$addToSet": "$artist_name"},
            "unique_tracks": {"$addToSet": "$spotify_track_uri"}
        }
    },
    {
        "$project": {
            "_id": 0,
            "year": "$_id",
            
            "total_streams": 1,
            "total_sec_played": 1,
            "listening_hours": {"$divide": ["$total_sec_played", 3600]},

            "unique_artists_played": {"$size": "$unique_artists"},
            "unique_tracks_played": {"$size": "$unique_tracks"},

            "skipped_count": 1,
            "skip_rate": {"$divide": ["$skipped_count", "$total_streams"]},

            "shuffled_count": 1,
            "shuffle_rate": {"$divide": ["$shuffled_count", "$total_streams"]}
        }
    },
    {
        "$merge": {
            "into": "yearSummaryMetrics",
            "whenMatched": "replace",
            "whenNotMatched": "insert"
        }
    }
])
# Ensure the year field is unique in the collection
db.yearSummaryMetrics.create_index("year", unique=True)
# Add count of new artists in each year
db.artistYearMetrics.aggregate([
    {
        "$match": {"is_new_artist": True}
    },
    {
        "$group": {
            "_id": "$year",
            "new_artists_count": {"$sum": 1}
        }
    },
    {
        "$project": {
            "year": "$_id", 
            "new_artists_count": 1, 
            "_id": 0
        }
    },
    {
        "$merge": {
            "into": "yearSummaryMetrics",
            "on": "year",
            "whenMatched": "merge",
            "whenNotMatched": "insert"
        }
    }
])
# Add count of new songs in each year
db.songYearMetrics.aggregate([
    {
        "$match": {"is_new_song": True}
    },
    {
        "$group": {
            "_id": "$year",
            "new_songs_count": {"$sum": 1}
        }
    },
    {
        "$project": {
            "year": "$_id", 
            "new_songs_count": 1, 
            "_id": 0
        }
    },
    {
        "$merge": {
            "into": "yearSummaryMetrics",
            "on": "year",
            "whenMatched": "merge",
            "whenNotMatched": "insert"
        }
    }
])
# Compute ratio of new artists to total unique artists played in each year
db.yearSummaryMetrics.aggregate([
    {
        "$set": {
            "new_artists_ratio": {
                "$cond": [
                    {"$eq": ["$unique_artists_played", 0]},
                    0,
                    {"$divide": ["$new_artists_count", "$unique_artists_played"]}
                ]
            }
        }
    },
    {
        "$merge": {
            "into": "yearSummaryMetrics",
            "on": "year",
            "whenMatched": "merge",
            "whenNotMatched": "insert"
        }
    }
])
# Compute ratio of new songs to total unique songs played in each year
db.yearSummaryMetrics.aggregate([
    {
        "$set": {
            "new_songs_ratio": {
                "$cond": [
                    {"$eq": ["$unique_tracks_played", 0]},
                    0,
                    {"$divide": ["$new_songs_count", "$unique_tracks_played"]}
                ]
            }
        }
    },
    {
        "$merge": {
            "into": "yearSummaryMetrics",
            "on": "year",
            "whenMatched": "merge",
            "whenNotMatched": "insert"
        }
    }
])

Make sure the previous steps were successful and that the database and the collections exists

In [9]:
# List all databases
databases = client.list_database_names()
# Check if a specific database exists
if db_name in databases:
    print(f"Database '{db_name}' exists.")

    for collection_name in db.list_collection_names():
        print(f"Collection '{collection_name}' exists.")
else:
    print(f"Database '{db_name}' does not exist.")

Database 'spotifyDatabase' exists.
Collection 'yearSummaryMetrics' exists.
Collection 'streamingRawData' exists.
Collection 'streamingAnalytics' exists.
Collection 'songYearMetrics' exists.
Collection 'artistYearMetrics' exists.


### 3. Export to CSV

After completing the database and collections, all analytical collections were exported to CSV files to facilitate Tableau visualization and sharing.

Collections exported:
- yearSummaryMetrics.csv: top-level yearly summary with listening hours, discovery metrics, skip/shuffle rates.
- artistYearMetrics.csv: artist-level yearly aggregates with engagement and discovery flags.
- songYearMetrics.csv: song-level yearly aggregates, including playback and engagement metrics.


In [12]:
import pandas as pd

collections = [
    "artistYearMetrics",
    "songYearMetrics",
    "yearSummaryMetrics"
]

for name in collections:
    data = list(db[name].find({}, {"_id": 0}))
    df = pd.DataFrame(data)
    df.to_excel(f"{name}.xlsx", index=False)

print("Export complete.")

Export complete.


## Tableau Dashboard

The Tableau dashboard is the primary interface for analyzing and visualizing listening behavior over the last decade. Using the exported CSVs, the dashboard allows exploration of trends in listening hours, discovery, engagement, and artist/song popularity, supporting the narrative of music identity evolution and exploration vs repetition.

Key features:
- Yearly overview of total streams, listening hours, and discovery ratios.
- Artist-level insights with skip rates, shuffle rates, and new artist tracking.
- Song-level analysis of engagement, first/last plays, and discovery trends.
- Filters for year, artist, and song to enable interactive exploration.

Link to dashboard: [Tableau Public Dashboard](https://public.tableau.com/app/profile/daniela.nieto.prada5218/viz/Spotify-Database/Sheet1)